# Export Eventhouse to Lakehouse — Counter_raw

This notebook queries a KQL database in an Eventhouse for recently ingested data
(`ingestion_time() > ago(1m)`) and writes the results to a **`Counter_raw`** Delta table
in the attached Lakehouse (Bronze layer, no transformations).

**Prerequisites:**
1. Attach this notebook to the **LH1** lakehouse in your target workspace (BCDR1 or BCDR2).
2. Ensure the Eventhouse KQL database is accessible from this workspace.
3. Update the `KUSTO_URI` and `DATABASE` variables in Cell 1 to match your environment.

**BCDR Workflow:**
- This notebook lives in BCDR1 initially, synced to BCDR2 via Git.
- Use `Switch-NotebookConnection.ps1` to swap the lakehouse attachment to BCDR2.
- Use `Manage-NotebookSchedule.ps1` to enable the schedule on BCDR2 and disable on BCDR1.

In [ ]:
# ── Cell 1: Configuration ──

# Eventhouse KQL endpoint — update these for your environment
KUSTO_URI   = "https://<your-eventhouse>.kusto.fabric.microsoft.com"
DATABASE    = "<your-kql-database>"

# KQL query: get rows ingested in the last 1 minute
KQL_QUERY   = "Counter | where ingestion_time() > ago(1m)"

# Target Delta table name in the attached Lakehouse
TARGET_TABLE = "Counter_raw"

print(f"Kusto URI : {KUSTO_URI}")
print(f"Database  : {DATABASE}")
print(f"Query     : {KQL_QUERY}")
print(f"Target    : {TARGET_TABLE}")

In [ ]:
# ── Cell 2: Query Eventhouse via Kusto Spark connector ──
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Use the Kusto Spark connector (pre-installed in Fabric Spark runtimes)
kusto_df = (
    spark.read
    .format("com.microsoft.kusto.spark.synapse.datasource")
    .option("spark.synapse.linkedService", "kustoDefault")  # Fabric auto-linked service
    .option("kustoCluster", KUSTO_URI)
    .option("kustoDatabase", DATABASE)
    .option("kustoQuery", KQL_QUERY)
    .option("authType", "ManagedIdentity")  # Use workspace managed identity
    .load()
)

row_count = kusto_df.count()
print(f"Rows returned: {row_count}")
kusto_df.show(10, truncate=60)

In [ ]:
# ── Cell 3: Write to Lakehouse as Delta table (Bronze — no transforms) ──
from pyspark.sql.functions import current_timestamp

if row_count == 0:
    print("No new rows to export. Skipping write.")
else:
    # Add an export timestamp for lineage tracking
    export_df = kusto_df.withColumn("_export_timestamp", current_timestamp())

    (export_df.write
     .format("delta")
     .mode("append")  # Append to preserve historical exports
     .saveAsTable(TARGET_TABLE))

    print(f"\u2713 {row_count} rows written to '{TARGET_TABLE}'")

In [ ]:
# ── Cell 4: Verify ──
total = spark.sql(f"SELECT COUNT(*) as total FROM {TARGET_TABLE}").collect()[0]["total"]
print(f"Total rows in {TARGET_TABLE}: {total}")

print(f"\nMost recent exports:")
spark.sql(f"""
    SELECT _export_timestamp, COUNT(*) as rows_exported
    FROM {TARGET_TABLE}
    GROUP BY _export_timestamp
    ORDER BY _export_timestamp DESC
    LIMIT 5
""").show()